# Lab-7 Encoder-Decoder

Aswin Singh Karki

ACE080BCT016


## Encoder-Decoder Architecture

The Encoder-Decoder architecture is a neural network design pattern primarily used for sequence-to-sequence (seq2seq) tasks, where the input and output sequences can have different lengths. It was introduced to address the limitations of traditional Recurrent Neural Networks (RNNs) in handling variable-length sequences, especially in tasks like machine translation, text summarization, and image captioning.

### Core Idea

The architecture consists of two main components:

1.  **Encoder**: Reads the input sequence and compresses it into a fixed-dimensional context vector (or thought vector).
2.  **Decoder**: Takes the context vector generated by the encoder and expands it into an output sequence.

### The Encoder

The Encoder is typically an RNN (like an LSTM or GRU) that processes the input sequence token by token. At each step, it updates its internal state. After processing the entire input sequence, the final hidden state(s) of the encoder are considered the "context vector." This vector is expected to capture the essence or a summary of the input sequence.

*   **Input**: A sequence of elements (e.g., words in a source language).
*   **Process**: Reads the entire input sequence and encodes it into a fixed-size vector representation.
*   **Output**: A context vector (also known as a thought vector or latent representation) that encapsulates the information of the input sequence.

### The Decoder

The Decoder is another RNN that generates the output sequence, one token at a time. It receives the context vector from the encoder as its initial hidden state. In each decoding step, it produces an output token and updates its own hidden state. This hidden state, along with the context vector, helps in generating the next token.

*   **Input**: The context vector from the encoder and previous output tokens (or a special start-of-sequence token).
*   **Process**: Generates the output sequence token by token, using the context vector and its own previous outputs/states.
*   **Output**: A sequence of elements (e.g., words in a target language).

### How Encoder-Decoder Addresses Traditional RNN Problems

1.  **Variable Length of Input and Output**: By separating the encoding and decoding processes, the Encoder-Decoder model inherently supports variable-length inputs and outputs. The encoder processes an input of any length into a fixed-size context vector, and the decoder generates an output of any desired length from that context vector.

2.  **Vanishing/Exploding Gradients**: While RNNs within the encoder and decoder still face these issues, the use of advanced RNN units like Long Short-Term Memory (LSTM) or Gated Recurrent Unit (GRU) cells significantly mitigates them. These units are designed with gates that regulate the flow of information, allowing the network to learn long-term dependencies.

3.  **Limited Context Window**: The context vector generated by the encoder attempts to summarize the entire input sequence. This allows the decoder to access information from the entire input, rather than just a limited window. However, for very long sequences, a single fixed-size context vector can become a bottleneck (information compression issue). This led to the development of "Attention Mechanisms" to further enhance context handling, allowing the decoder to selectively focus on relevant parts of the input sequence at each decoding step.

### A Simple Encoder-Decoder Example with Keras/TensorFlow

Let's illustrate a very basic Encoder-Decoder model for a simple sequence-to-sequence task, like reversing a sequence of numbers. This example will use Keras with a functional API, which is suitable for more complex models than the sequential API.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# Define the input and output sequence lengths
INPUT_SEQUENCE_LENGTH = 10
OUTPUT_SEQUENCE_LENGTH = 10
NUM_FEATURES = 1 # We'll just reverse a sequence of single numbers

# Generate some dummy data
def generate_data(num_samples, seq_len, num_features):
    X = np.random.randint(0, 10, size=(num_samples, seq_len, num_features))
    y = np.flip(X, axis=1) # Reverse the sequence for output
    return X, y

num_samples = 10000
X_train, y_train = generate_data(num_samples, INPUT_SEQUENCE_LENGTH, NUM_FEATURES)

print("Example input:")
print(X_train[0].flatten())
print("Example output (reversed input):")
print(y_train[0].flatten())

Example input:
[7 5 1 4 4 7 4 2 8 1]
Example output (reversed input):
[1 8 2 4 7 4 4 1 5 7]


In [2]:
# Define the Encoder
encoder_inputs = Input(shape=(INPUT_SEQUENCE_LENGTH, NUM_FEATURES))
encoder_lstm = LSTM(128, return_state=True) # 128 is the number of LSTM units
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

# Define the Decoder
decoder_inputs = Input(shape=(OUTPUT_SEQUENCE_LENGTH, NUM_FEATURES))
# We set up our decoder to return full output sequences, and to return internal states as well.
# We don't use the return states in the training model, but will use them in inference.
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(NUM_FEATURES, activation='relu')
decoder_outputs = decoder_dense(decoder_outputs)

# Define the full model for training
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile the model
model.compile(optimizer='adam', loss='mse') # Mean Squared Error for numerical sequences

# Display model summary
model.summary()

# Train the model
# For training, the decoder input needs to be the shifted version of the target sequence
# For sequence reversal, the input to the decoder is essentially the 'start' sequence
# (or some placeholder) followed by the previously generated number.
# For simplicity, we'll feed the actual target sequence to the decoder during training
# (this is common in 'teacher forcing').
# In a real scenario, decoder_inputs would be lagged versions of y_train or all zeros
# followed by the first element of y_train if predicting character by character.
# For this numerical sequence reversal, feeding y_train directly is a valid simplification.
# A more robust teacher-forcing approach would shift y_train by one timestep.

# Create a dummy decoder input that mirrors the target output structure
# In this simple case, since the output is just a reversed sequence of numbers,
# we can use the target sequence itself for decoder_inputs during training (teacher forcing).
# For sequence generation, typically you would have a start-of-sequence token
# and feed the previous predicted output as input for the next step.
# Here, we just want the decoder to learn to output the reversed sequence directly.

model.fit([X_train, y_train], y_train, batch_size=64, epochs=100, validation_split=0.2)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 10, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     66,560 │ input_layer[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 10, 128), │     66,560 │ input_layer_1[0]… │
│                     │ (None, 128),      │            │ lstm[0][1],       │
│                     │ (None, 128)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10, 1)     │        129 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 133,249 (520.50 KB)

 Trainable params: 133,249 (520.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - loss: 4.5361 - val_loss: 0.0637
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - loss: 0.0227 - val_loss: 0.0101
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - loss: 0.0067 - val_loss: 0.0042
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - loss: 0.0030 - val_loss: 0.0021
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - loss: 0.0015 - val_loss: 0.0011
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - loss: 8.9587e-04 - val_loss: 7.0819e-04
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - loss: 5.8959e-04 - val_loss: 5.0967e-04
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - loss: 4.4227e-04 - val_loss: 3.5059e-04
Epoch 9/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - loss: 3.4839e-04 - val_loss: 3.2389e-04
Epoch 10/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - loss: 2.8069e-04 - val_loss: 2.6869e-04
Epoch 11/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - loss: 2.6433e-04 - va

### Teacher Forcing

**Teacher forcing** is a technique used during the training of recurrent neural networks (RNNs) for sequence generation tasks, like in this Encoder-Decoder model. Instead of feeding the decoder's own predicted output from the previous timestep as input to generate the next token, the actual (correct) target token from the training data is fed as input. This helps stabilize training and allows the model to learn faster, especially in the early stages, by providing consistent correct guidance.

### Inference Model (Prediction)

To make predictions, we need to define separate inference models for the encoder and decoder. The encoder model will output the context vector (states), and the decoder model will take these states and generate the output sequence one step at a time.

### Encoder-Decoder Example with PyTorch

Now, let's explore another implementation of the Encoder-Decoder architecture using PyTorch. This example will focus on a more typical sequence-to-sequence task, such as machine translation (English to French), to illustrate how these models are built and trained in a different framework. It will cover data preparation, defining the Encoder and Decoder RNNs, and setting up a training and evaluation loop.

In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [6]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [21]:
PATH = r"C:\Users\aswin\Desktop\AI Lab\Lab7\eng-fra.txt"
input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['nous sommes impitoyables', 'we re ruthless']


15

In [22]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [23]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

In [24]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

## Training Loop

In [25]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [26]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [27]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [28]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [29]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [30]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

## Training and Evaluating

In [31]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 24s (- 15m 41s) (5 2%) 1.9562
0m 33s (- 10m 44s) (10 5%) 1.2730
0m 45s (- 9m 23s) (15 7%) 1.0562
0m 59s (- 8m 58s) (20 10%) 0.9141
1m 10s (- 8m 10s) (25 12%) 0.8037
1m 19s (- 7m 31s) (30 15%) 0.7196
1m 29s (- 7m 2s) (35 17%) 0.6404
1m 38s (- 6m 35s) (40 20%) 0.5708
1m 47s (- 6m 10s) (45 22%) 0.5045
1m 56s (- 5m 49s) (50 25%) 0.4437
2m 5s (- 5m 29s) (55 27%) 0.3941
2m 14s (- 5m 13s) (60 30%) 0.3461
2m 23s (- 4m 58s) (65 32%) 0.3046
2m 32s (- 4m 42s) (70 35%) 0.2696
2m 41s (- 4m 28s) (75 37%) 0.2406
2m 49s (- 4m 14s) (80 40%) 0.2128
2m 59s (- 4m 3s) (85 42%) 0.1905
3m 8s (- 3m 50s) (90 45%) 0.1689
3m 17s (- 3m 38s) (95 47%) 0.1507
3m 26s (- 3m 26s) (100 50%) 0.1386
3m 35s (- 3m 14s) (105 52%) 0.1260
3m 44s (- 3m 3s) (110 55%) 0.1144
3m 52s (- 2m 52s) (115 57%) 0.1035
4m 1s (- 2m 41s) (120 60%) 0.0989
4m 10s (- 2m 30s) (125 62%) 0.0921
4m 19s (- 2m 19s) (130 65%)

In [33]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> nous sommes en retard
= we re behind schedule
< we re late <EOS>

> nous sommes chez nous
= we re home
< we re home <EOS>

> tu es mon amie
= you are my friend
< you are my friend <EOS>

> je suis credule
= i m gullible
< she is very beautiful <EOS>

> elles s amusent
= they re enjoying themselves
< he is my brother <EOS>

> vous etes tres riches
= you are very rich
< you are very rich <EOS>

> vous etes intrepides
= you re fearless
< you aren t looking <EOS>

> je suis si malade
= i am so sick
< i am so sick <EOS>

> il est en faillite
= he is bankrupt
< he is bankrupt <EOS>

> je suis obstine
= i m stubborn
< she s very busy <EOS>



### Discussion and Conclusion

This notebook introduced the Encoder-Decoder architecture for sequence-to-sequence tasks. We demonstrated its core components (encoder and decoder) and how it handles variable-length sequences, offering a conceptual improvement over basic RNNs. A simple Keras/TensorFlow example showcased its application in sequence reversal, highlighting the fundamental principles. While this example is basic, it lays the groundwork for understanding more complex models incorporating features like attention mechanisms in real-world applications.